# Telco Customer Churn — Exploratory Data Analysis

**Business problem.** A telecom provider loses roughly 1 in 4 customers. Acquiring a new customer costs
several times more than retaining an existing one, so the retention team wants to know *which* customers
are likely to churn and *why*, early enough to intervene with targeted offers.

**This notebook** audits data quality, explores churn drivers, and records the cleaning decisions that
the production pipeline (`src/data.py`) implements. The modeling itself lives in `src/train.py` so the
exact same code runs in CI and in the Docker image — the notebook is for narrative, not production logic.

**Data.** IBM sample dataset via [Kaggle](https://www.kaggle.com/datasets/blastchar/telco-customer-churn):
7,043 customers, 21 columns covering demographics, subscribed services, contract and billing details,
and the churn label (did the customer leave within the last month).

In [ ]:
import sys
sys.path.append("..")  # so the notebook imports the same code the pipeline uses

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src.data import load_raw, clean

sns.set_theme(style="whitegrid", palette="muted")
pd.set_option("display.max_columns", None)

## 1. Load and first look

In [ ]:
df_raw = load_raw("../data/WA_Fn-UseC_-Telco-Customer-Churn.csv")
print(df_raw.shape)
df_raw.head()

In [ ]:
df_raw.info()

## 2. Data quality audit

Three issues surface immediately, and each gets an explicit, documented decision rather than a silent fix.

### 2.1 `TotalCharges` is text, not numeric

`info()` shows `TotalCharges` as `object`. Coercing reveals why:

In [ ]:
blank_mask = df_raw["TotalCharges"].str.strip() == ""
print(f"Blank TotalCharges rows: {blank_mask.sum()}")
df_raw.loc[blank_mask, ["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]]

All 11 blanks belong to customers with **tenure = 0** — brand new accounts that have not been billed yet.
These are not missing values; the true amount charged so far is zero.

**Decision: impute 0**, not the median and not `MonthlyCharges`, and do not drop the rows — new customers
are exactly the population a churn model must handle at serving time.

### 2.2 `SeniorCitizen` is 0/1 while every other binary column is Yes/No

**Decision: map to Yes/No** so plots read consistently and one encoder handles all categoricals.

### 2.3 No duplicate customers (verified below), so no dedup step is needed.

In [ ]:
assert df_raw["customerID"].duplicated().sum() == 0
df = clean(df_raw)  # applies the two decisions above; same function the pipeline calls
df[["TotalCharges", "SeniorCitizen"]].dtypes

## 3. Target balance

Churn is imbalanced. This shapes everything downstream: stratified splits, class weighting,
and refusing to treat accuracy as the headline metric (predicting "No" for everyone scores 73%).

In [ ]:
churn_rate = (df["Churn"] == "Yes").mean()
print(f"Churn rate: {churn_rate:.1%}")

fig, ax = plt.subplots(figsize=(4, 3))
df["Churn"].value_counts().plot.bar(ax=ax, color=["#4c72b0", "#c44e52"])
ax.set_title("Class balance"); ax.set_ylabel("Customers")
plt.tight_layout(); plt.show()

## 4. Churn drivers

### 4.1 Contract type — the strongest single signal

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
rates = df.groupby("Contract", observed=True)["Churn"].apply(lambda s: (s == "Yes").mean()).sort_values()
rates.plot.barh(ax=ax, color="#c44e52")
ax.set_title("Churn rate by contract type"); ax.set_xlabel("Churn rate")
for i, v in enumerate(rates): ax.text(v, i, f" {v:.0%}", va="center")
plt.tight_layout(); plt.show()

Month-to-month customers churn at roughly **10x** the rate of two-year contracts. Contract length is
partly a proxy for commitment, but it is also the most actionable lever the retention team owns:
converting a high-risk month-to-month customer to an annual plan directly attacks the risk factor.

### 4.2 Tenure — churn is front-loaded

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
sns.histplot(data=df, x="tenure", hue="Churn", multiple="fill", bins=24, ax=ax,
             palette={"No": "#4c72b0", "Yes": "#c44e52"})
ax.set_title("Share of churners by tenure (months)"); ax.set_ylabel("Proportion")
plt.tight_layout(); plt.show()

Most churn happens in the first year. Retention effort has the highest expected value in the
onboarding window, not spread evenly across the customer base.

### 4.3 Monthly charges and service mix

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
sns.kdeplot(data=df, x="MonthlyCharges", hue="Churn", common_norm=False, fill=True, ax=axes[0],
            palette={"No": "#4c72b0", "Yes": "#c44e52"})
axes[0].set_title("Monthly charges by churn")

svc = df.groupby("InternetService", observed=True)["Churn"].apply(lambda s: (s == "Yes").mean())
svc.plot.bar(ax=axes[1], color="#c44e52", rot=0)
axes[1].set_title("Churn rate by internet service"); axes[1].set_ylabel("Churn rate")
plt.tight_layout(); plt.show()

Churners skew toward **higher monthly charges**, and **fiber optic** customers churn far more than
DSL — likely a price/expectation problem rather than a product one, since fiber is the premium tier.

### 4.4 Payment method

In [ ]:
pay = df.groupby("PaymentMethod", observed=True)["Churn"].apply(lambda s: (s == "Yes").mean()).sort_values()
fig, ax = plt.subplots(figsize=(6.5, 3))
pay.plot.barh(ax=ax, color="#c44e52")
ax.set_title("Churn rate by payment method"); ax.set_xlabel("Churn rate")
plt.tight_layout(); plt.show()

Electronic check stands out at roughly double the churn of automatic payment methods — manual monthly
payment is a monthly decision point to cancel, whereas autopay removes the friction of staying.

## 5. Conclusions

**Data quality decisions** (implemented in `src/data.py`, enforced by `tests/test_data.py`):
1. `TotalCharges` blanks → impute 0 (they are unbilled new customers, not missing data)
2. `SeniorCitizen` 0/1 → Yes/No for encoding consistency
3. No duplicates; no dedup needed

**Risk profile of a churner:** month-to-month contract, short tenure, fiber optic internet,
high monthly charges, paying by electronic check, no tech support or online security add-ons.

**Modeling implications:** stratified splits and class weighting for the 26.5% positive class;
ROC-AUC and PR-AUC over accuracy; recall on churners matters most because a missed churner costs
more than a wasted retention offer.

**Next:** `src/train.py` trains the candidate models, `src/api.py` serves the winner,
and `docs/DEPLOYMENT_AWS.md` covers taking it to production.